# 02 Interactive display
Sensors, impacts and virtual points can be added and moved in the 3D view interactively. The objects can then be moved around in the 3D view. The `pyFBS` supports also snapping of the objects to the surface of a predefined mesh (ussualy obtained from a STL file). When object snaps to the surface, not only the position of the object changes, but also the orientation of the object alligns with the normal of the mesh at the intersection (this feature can also be disabled). 

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys
sys.path.append(r"C:\Users\tomaz.bregar\Desktop\git_projects\pyFBS")

In [3]:
import pyFBS

import pandas as pd

## 3D View
Open 3Dviewer in the background.

In [4]:
view3D = pyFBS.view3D()

#### Download example files

In [5]:
pyFBS.download_lab_testbench()

100%|██████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 4700.63it/s]

#### Add a structure
Load an example laboratory testbench and add a mesh to the 3D view.

In [6]:
stl = r"./lab_testbench/STL/A.stl"
mesh = view3D.add_stl(stl,name = "ts",color = "#83afd2")

#### Datasets
Load a predefined datasets from an example. 

In [7]:
pos_xlsx = r"./lab_testbench/Measurements/AM_measurements.xlsx"

df_sensors = pd.read_excel(pos_xlsx, sheet_name='Sensors_A')
df_impacts = pd.read_excel(pos_xlsx, sheet_name='Impacts_A')
df_vp = pd.read_excel(pos_xlsx, sheet_name='VP_Channels')

## Sensors
To enable interaction of sensors in the 3D view, just simply call a function `view3D.add_acc_dynamic(mesh,predefined = df_sensors)`. This will place the predefined sensors in the display and enable interaction with them and will allow you to add additional sensors. If you are starting a completely new analysis, you don't need the predefined data you can simply start with an empty dataset (i.e. `predefined = None`).

The object can be moved around by moving a black sphere in the 3D view. Arbitrary rotation around each local axis can be obtained my moving colored spheres (red - rotation around *X*, green - rotation around *Y*, blue - rotation around *Z*)

In [8]:
def angle(vector1, vector2):
    """
    Calculates angle of rotation between two 3D vectors.

    :param vector1: 3D vector
    :type vector1: array(float)
    :param vector2: 3D vector
    :type vector2: array(float)
    :return: angle
    """

    v1_u = unit_vector(vector1)
    v2_u = unit_vector(vector2)
    minor = np.linalg.det(np.stack((v1_u[-2:], v2_u[-2:])))
    if minor == 0:
        sign = 1
    else:
        sign = -np.sign(minor)
    dot_p = np.dot(v1_u, v2_u)
    dot_p = min(max(dot_p, -1.0), 1.0)
    return sign * np.arccos(dot_p)

def unit_vector(vector):
    """
    Returns the unit vector of input vector.

    :param vector: A 3D "source" vector
    :type vector: array(float)
    :return unit vector:
    """

    return vector / np.linalg.norm(vector)

def M(axis, theta):
    """
    Calculates rotational matrix based on the Euler-Rodrigues formula.

    :param axis: Axis of rotation
    :type axis: array(float)
    :param theta: Angle of rotation
    :type theta: float
    :return: Rotational matrix
    """
    t = expm(cross(eye(3), axis / norm(axis) * (theta)))
    return t

import matplotlib.pyplot as plt
import numpy as np
from numpy import cross, eye
from scipy.linalg import expm, norm
import pandas as pd
from scipy.spatial.transform import Rotation as R
from pyts.decomposition import SingularSpectrumAnalysis


In [9]:
M(np.asarray([0,0.75,75]),1)

array([[ 0.54030231, -0.84142891,  0.00841429],
       [ 0.84142891,  0.54034827,  0.00459652],
       [-0.00841429,  0.00459652,  0.99995403]])

In [10]:
vec1 = [0,0,1]
vec2 = [0,1,0]
    
angle(vec1,vec2)
        

1.5707963267948966

In [11]:
view3D.fixed_rotation = None

In [12]:
view3D.add_acc_dynamic(mesh,predefined = df_sensors,fixed_rotation = 10)

Additonaly, fixed rotation angle can be defined by passing `fixed_rotation` variable when adding dynamic sensors in the display. After clicking on the sphere widget the sensor will rotate for the predetermined angle, based on the sign (clock or counterclockwise). 

In [13]:
view3D.all_accs_dynamic[0].fixed_theta

10

In [14]:
view3D.add_acc_dynamic(mesh,predefined = df_sensors)

The position and orientation data can be obtained by simply calling a function `view3D.get_acc_data()`:  

In [ ]:
df_acc_updated = view3D.get_acc_data()
df_acc_updated

From the new positions and orientations of sensors a channel dataset can be generated (currently all the accelerometers are considered as tri-axial). If you have uni-axial accelerometers, redundant channels can be discarded afterwards.

In [ ]:
df_chn_updated = pyFBS.utility.generate_channels_from_sensors(df_acc_updated)
df_chn_updated

If you have the channel dataset (tri-axial) you can generate the sensor dataset. This transformation is not unique and gimbal lock problem can arise. In this case third angle (rotation around Z axis) is set to zero and a warning is raised (see [scipy.spatial.transform.Rotation.as_euler](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.transform.Rotation.as_euler.html) for more details). Nevertheless, the obtained rotation angles still represent the correct rotation.

In [ ]:
df_acc_from_chn = pyFBS.utility.generate_sensors_from_channels(df_chn_updated)
df_acc_from_chn

## Impacts

Adding impacts interactively to the 3D view is the same, only the object display is different.

In [ ]:
view3D.add_imp_dynamic(mesh,predefined = df_impacts)

The updated positions and orientation of the virtual points can be obtained directly:

In [ ]:
df_imp_updated = view3D.get_imp_data()
df_imp_updated

## Virtual points

Adding virtual points interactively to the 3D view is the same, only the object display is different.

In [ ]:
view3D.add_vp_dynamic(mesh,predefined = df_vp)

The updated positions and orientation of the virtual points can be obtained directly:

In [ ]:
df_vp_updated = view3D.get_vp_data()
df_vp_updated

## Output
You can save the new positions and orientations of the objects in Excel file in a simple manner:  

In [ ]:
# with pd.ExcelWriter('./output_file.xlsx') as writer:  
#     df_acc_updated.to_excel(writer, sheet_name='Sensors',index = False)
#     df_imp_updated.to_excel(writer, sheet_name='Impacts',index = False)
#     df_chn_updated.to_excel(writer, sheet_name='Channels',index = False)